In [56]:
import os
import joblib
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.preprocessing import MinMaxScaler
from scipy.optimize import minimize

plt.style.use("dark_background")

In [57]:
quad_lin_reg_model_storage_path = os.path.join(
    os.path.dirname(os.getcwd()), "..", "data", "models", "quad_lin_reg"
)

X =joblib.load(os.path.join(quad_lin_reg_model_storage_path, "X.npy"))
model = joblib.load(os.path.join(quad_lin_reg_model_storage_path, "linear_model.pkl"))
poly = joblib.load(os.path.join(quad_lin_reg_model_storage_path, "poly_features.pkl"))
X_quad = joblib.load(os.path.join(quad_lin_reg_model_storage_path, "X_quad.npy"))
y_pred = joblib.load(os.path.join(quad_lin_reg_model_storage_path, "y_pred.npy"))
y_label = joblib.load(os.path.join(quad_lin_reg_model_storage_path, "y_label.npy"))

processed_data_path = os.path.join(
    os.path.dirname(os.getcwd()), "..", "data", "processed"
)
scaler = joblib.load(os.path.join(processed_data_path, "scaler.pkl"))

In [ ]:
def objective(x):
    x = np.array(x).reshape(1, -1)           # shape (1,3)
    x_quad = poly.transform(x)               # quadratic features
    y_pred = model.predict(X_quad)[0, 0]     # single output
    return -y_pred 

bounds = [(X[:,0].min(), X[:,0].max()),
          (X[:,1].min(), X[:,1].max()),
          (X[:,2].min(), X[:,2].max())]

# Try multiple initial guesses
best_val = -np.inf
best_x = None
for x0 in [X.mean(axis=0), X.min(axis=0), X.max(axis=0)]:
    res = minimize(objective, x0, bounds=bounds, method='L-BFGS-B')
    if -res.fun > best_val:
        best_val = -res.fun
        best_x = res.x

# print("Optimal input:", best_x)
# print("Predicted maximum output:", best_val)

x_full = np.zeros(14)        # 14 features expected
x_full[5] = best_x[0]          # put optimized features in the right place
x_full[7] = best_x[1]          # put optimized features in the right place
x_full[13] = best_x[2]          # put optimized features in the right place
# optionally fill other 11 with their mean values in scaled space
# x_full[3:] = 0.5  # (since min-max scaled, mean is ~0.5)

best_x_original = scaler.inverse_transform(x_full.reshape(1, -1))[0]

print("Optimal Flow_C", best_x_original[5])
print("Optimal Flow_P:", best_x_original[7])
print("Optimal Current_Limit_P:", best_x_original[13])
print("Maximum Removal:", best_val * 100)

Optimal Flow_C 0.5365384615384615
Optimal Flow_P: 0.5826923076923076
Optimal Current_Limit_P: 48.07692307692307


visualize minima/maxima in plot/3dplot

denormalize values